# RAG with DSPy and ChromaDB

Your model can sound confident about Linux kernel memory — but was any of that in your docs?

In Introduction to RAG we covered retrieval-augmented generation; in [14_chromadb.ipynb](14_chromadb.ipynb) and [15_chunking.ipynb](15_chunking.ipynb) we built vector search with ChromaDB. Here we wire that retriever into a **DSPy program** and measure whether each change is worth it.

## Three phases

We build and benchmark the same tech-QA task three times:

1. **DSPy Modules** — `ChainOfThought` answers from the LM alone.
2. **Adding RAG** — `ChromadbRM` retrieves passages, then the LM generates from context.
3. **Optimized RAG** — automatic prompt tuning with MIPROv2 on the RAG program.

After each phase we record **accuracy**, **cost**, and **latency**, split into **train-time** (optimization) and **inference-time** (dev-set evaluation), normalized to our actual workload (300 dev questions).

## What you'll learn

- Plug a ChromaDB retriever into DSPy via `ChromadbRM` ([ext/chromadb_rm.py](ext/chromadb_rm.py))
- Evaluate programs on accuracy, cost, and latency — not vibes
- Compare baseline, RAG, and optimized-RAG pipelines on the same dev set

## Setup (and imports)

### Load environment variables

Same pattern as [12_dspy_evaluation_and_optimization.ipynb](12_dspy_evaluation_and_optimization.ipynb).

In [1]:
import os

try:
    from google.colab import userdata

    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path(".env").exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv

    load_dotenv(override=True)

### Connect a language model

In [2]:
import dspy  # https://dspy.ai/

lm = dspy.LM(
    model="openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
dspy.configure(lm=lm)

### Imports and paths

Optional tracing: see [mlflow.md](mlflow.md).

In [3]:
import random
import time
from pathlib import Path

import chromadb  # https://docs.trychroma.com/
import orjson  # https://github.com/ijl/orjson
import pandas as pd
from dspy.evaluate import SemanticF1
from dspy.utils import download
from IPython.display import display

from ext.benchmark_utils import benchmark_program, results_to_frame
from ext.chromadb_rm import ChromadbRM

DATA_DIR = Path("data") / "ragqa"
CHROMA_DIR = DATA_DIR / "chroma"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 5
MAX_CHARS = 6000
BATCH_SIZE = 5000

## Ungrounded LLM Answers

**Phase 1.** We already know `ChainOfThought` from [08_dspy_changing_modules.ipynb](08_dspy_changing_modules.ipynb). Here it answers tech questions with no retrieval — pure parametric knowledge.

In [4]:
cot = dspy.ChainOfThought("question -> response")

demo = cot(question="what are high memory and low memory on linux?")
print(demo.response)

In Linux, "low memory" generally refers to the first portion of RAM that is directly accessible by the kernel and applications, typically the lower addresses in memory, often within the first 4GB on 32-bit systems. "High memory" refers to additional memory above this range that is not directly mapped into the kernel's address space, hence requiring special handling to access. On 64-bit systems, this distinction is less significant because the entire RAM can usually be directly addressed, but the terms may still be used in specific contexts such as kernel memory management.


## Using DSPy well involves evaluation and iterative development.

You already know a lot about DSPy at this point. If all you want is quick scripting, this much of DSPy already enables a lot. Sprinkling DSPy signatures and modules into your Python control flow is a pretty ergonomic way to just get stuff done with LMs.

That said, you're likely here because you want to build a high-quality system and improve it over time. The way to do that in DSPy is to iterate fast by evaluating the quality of your system and using DSPy's powerful tools, e.g. Optimizers.

> [`MIPROv2`](https://dspy.ai/api/optimizers/MIPROv2/) (Multiprompt Instruction PRoposal Optimizer Version 2) is an prompt optimizer capable of optimizing both instructions and few-shot examples jointly. It does this by bootstrapping few-shot example candidates, proposing instructions grounded in different dynamics of the task, and finding an optimized combination of these options using Bayesian Optimization. It can be used for optimizing few-shot examples & instructions jointly, or just instructions for 0-shot optimization.

## Manipulating Examples in DSPy.

We need labeled `(question, response)` pairs and a metric. This notebook uses the [RAG-QA Arena Tech](https://arxiv.org/abs/2407.13998) dataset — StackExchange questions with gold answers.

- **Train** (200): passed to MIPROv2
- **Dev** (300): our evaluation workload — all benchmark numbers use this split
- **Test** (500): held out for final checks

In [5]:
examples_url = "https://huggingface.co/dspy/cache/resolve/main/ragqa_arena_tech_examples.jsonl"
examples_path = DATA_DIR / "ragqa_arena_tech_examples.jsonl"

if not examples_path.exists():
    download(examples_url)
    Path("ragqa_arena_tech_examples.jsonl").rename(examples_path)

with examples_path.open("rb") as f:
    raw = [orjson.loads(line) for line in f]

data = [
    dspy.Example(**row).with_inputs("question")
    for row in raw
]

random.Random(0).shuffle(data)
trainset, devset, testset = data[:200], data[200:500], data[500:1000]

len(trainset), len(devset), len(testset)

(200, 300, 500)

In [6]:
example = devset[2]
example

Example({'question': 'how to chmod without /usr/bin/chmod?', 'response': "Run the loader directly, and pass it the command you want to run: `/lib/ld-linux.so /bin/chmod +x /bin/chmod`. \nThe exact path might change, especially on a 64-bit system, so version is named something like `/lib64/ld-linux-x86-64.so.2.` \nAlternatively, if busybox is installed, you can execute `busybox chmod +x /bin/chmod`. \nAnother hack is the command: `mv /bin/chmod /bin/chmod.orig cp -a /bin/chown /bin/chmod`. \nYou can also prepare another executable file, copy chmod over it to maintain executable permissions, `$ cp /bin/ls chmod $ cp /bin/chmod`, or use the install utility to do this with permission settings in one step; the command for this would be: `$ install -m a+x /bin/chmod . $ ./chmod # executes'`. \nPiping contents into an already executable file is another option (`cp /usr/bin/executable_file ~/executable_file cat /usr/bin/chmod > ~/executable_file ~/executable_file +x file_to_be_executed.sh`), a

## Evaluation in DSPy.

What kind of metric can suit our question-answering task? There are many choices, but since the answers are long, we may ask:

1. How well does the system response _cover_ all key facts in the gold response?
2. And the other way around, how well is the system response _not saying things_ that aren't in the gold response?

That metric is essentially a **semantic F1**, so let's load a `SemanticF1` metric from DSPy. This metric is actually implemented as a [very simple DSPy module](https://github.com/stanfordnlp/dspy/blob/main/dspy/evaluate/auto_evaluation.py#L21) using whatever LM we're working with.

### Scoring a single prediction

In [11]:
from dspy.evaluate import SemanticF1

# Instantiate the metric.
metric = SemanticF1(decompositional=True)

# Produce a prediction from our `cot` module, using the `example` above as input.
pred = cot(**example.inputs())

# Compute the metric score for the prediction.
score = metric(example, pred).score

print(f"Question: \t {example.question}\n")
print(f"Gold Response: \t {example.response}\n")
print(f"Predicted Response: \t {pred.response}\n")
print(f"Semantic F1 Score: {score:.2f}")

Question: 	 how to chmod without /usr/bin/chmod?

Gold Response: 	 Run the loader directly, and pass it the command you want to run: `/lib/ld-linux.so /bin/chmod +x /bin/chmod`. 
The exact path might change, especially on a 64-bit system, so version is named something like `/lib64/ld-linux-x86-64.so.2.` 
Alternatively, if busybox is installed, you can execute `busybox chmod +x /bin/chmod`. 
Another hack is the command: `mv /bin/chmod /bin/chmod.orig cp -a /bin/chown /bin/chmod`. 
You can also prepare another executable file, copy chmod over it to maintain executable permissions, `$ cp /bin/ls chmod $ cp /bin/chmod`, or use the install utility to do this with permission settings in one step; the command for this would be: `$ install -m a+x /bin/chmod . $ ./chmod # executes'`. 
Piping contents into an already executable file is another option (`cp /usr/bin/executable_file ~/executable_file cat /usr/bin/chmod > ~/executable_file ~/executable_file +x file_to_be_executed.sh`), as is utilizi

### Scoring on the `devset`

For evaluation, you could use the metric above in a simple loop and just average the score. But for nice parallelism and utilities, we can rely on `dspy.Evaluate`.

Our [`benchmark_program`](ext/benchmark_utils.py#L18) helper records three aspects on the **actual dev workload**:

- **Accuracy** — SemanticF1 on all 300 dev examples
- **Cost** — USD from `lm.history` (LiteLLM estimates)
- **Latency** — wall-clock seconds **per query**

### Phase 1 benchmark — inference

In [12]:
benchmark_rows = []

row_p1 = benchmark_program(
    cot,
    devset=devset,
    metric=metric,
    phase="1 baseline CoT",
    mode="inference",
    lm=lm,
    num_threads=24,
)
benchmark_rows.append(row_p1)
results_to_frame(benchmark_rows)

Average Metric: 197.36 / 300 (65.8%): 100%|██████████| 300/300 [01:41<00:00,  2.97it/s]

2026/06/20 11:39:50 INFO dspy.evaluate.evaluate: Average Metric: 197.3632886691797 / 300 (65.8%)


,question,example_response,gold_doc_ids,reasoning,pred_response,SemanticF1
0,"when to use c over c++, and c++ over c?","If you are equally familiar with both C++ and C, it's advisable to...",[733],"C is a procedural programming language that provides a simple, eff...",Use C over C++ primarily when you need a lightweight language with...,✔️ [Prediction(\n score=0.7741935483870969\n)]
1,should images be stored in a git repository?,"One viewpoint expresses that there is no significant downside, esp...","[6253, 6254, 6275, 6278, 8215]",Storing images in a git repository depends on the context. For sma...,Images should generally not be stored directly in a git repository...,✔️ [Prediction(\n score=0.75\n)]


,phase,mode,n_queries,accuracy_pct,cost_total_usd,cost_per_query_usd,latency_sec_per_query,wall_seconds
0,1 baseline CoT,inference,300,65.79,0.111354,0.000371,0.337462,101.238555


<details>
<summary>Tracking Evaluation Results in MLflow Experiment</summary>

<br/>

To track and visualize the evaluation results over time, you can record the results in MLflow Experiment.


```python
import mlflow

with mlflow.start_run(run_name="rag_evaluation"):
    evaluate = dspy.Evaluate(
        devset=devset,
        metric=metric,
        num_threads=24,
        display_progress=True,
    )

    # Evaluate the program as usual
    result = evaluate(cot)


    # Log the aggregated score
    mlflow.log_metric("semantic_f1_score", result.score)
    # Log the detailed evaluation results as a table
    mlflow.log_table(
        {
            "Question": [example.question for example in eval_set],
            "Gold Response": [example.response for example in eval_set],
            "Predicted Response": [output[1] for output in result.results],
            "Semantic F1 Score": [output[2] for output in result.results],
        },
        artifact_file="eval_results.json",
    )
```

To learn more about the integration, visit [MLflow DSPy Documentation](https://mlflow.org/docs/latest/llms/dspy/index.html) as well.

</details>

So far, we built a chain-of-thought module for question answering and evaluated it on the dev set.

Can we do better? Next we add **retrieval-augmented generation** with ChromaDB, benchmark that lift, then use a DSPy Optimizer to _compile_ the RAG program to higher-quality prompts.

## Basic Retrieval-Augmented Generation (RAG).

**Phase 2.** We add retrieval — the pattern from [Introduction to RAG](13_rag.typ): retrieve relevant passages, augment the prompt, then generate.

Corpus: downsampled RAG-QA Tech corpus (~28k documents). Documents are already passage-sized; no re-chunking needed here.

In [13]:
corpus_url = "https://huggingface.co/dspy/cache/resolve/main/ragqa_arena_tech_corpus.jsonl"
corpus_path = DATA_DIR / "ragqa_arena_tech_corpus.jsonl"

if not corpus_path.exists():
    download(corpus_url)
    Path("ragqa_arena_tech_corpus.jsonl").rename(corpus_path)

### Set up your system's retriever.

We use **`ChromadbRM`** — a `dspy.Retrieve` subclass backed by ChromaDB. 

We will embed locally with **`all-MiniLM-L6-v2`** at no API cost.

In [14]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(name="ragqa_tech")

if collection.count() == 0:
    with corpus_path.open("rb") as f:
        corpus = [
            orjson.loads(line)["text"][:MAX_CHARS]
            for line in f
        ]
    print(f"Loaded {len(corpus)} documents. Ingesting into Chroma…")

    for start in range(0, len(corpus), BATCH_SIZE):
        batch = corpus[start : start + BATCH_SIZE]
        batch_ids = [str(start + i) for i in range(len(batch))]
        collection.add(
            ids=batch_ids,
            documents=batch,
        )
        print(f"  ingested {start + len(batch)} / {len(corpus)}")

Loaded 28436 documents. Ingesting into Chroma…
  ingested 5000 / 28436
  ingested 10000 / 28436
  ingested 15000 / 28436
  ingested 20000 / 28436


KeyboardInterrupt: 

In [15]:
print(f"Collection count: {collection.count()}")

Collection count: 20000


In [16]:
search = ChromadbRM(
    collection_name="ragqa_tech",
    persist_directory=str(CHROMA_DIR),
    k=TOP_K,
)

demo_retrieval = search("what are high memory and low memory on linux?")
for i, passage in enumerate(demo_retrieval.passages, start=1):
    preview = passage[:120] + ("…" if len(passage) > 120 else "")
    print(f"[{i}] {preview}")

[1] This is relevant to the Linux kernel; Im not sure how any Unix kernel handles this. The High Memory is the segment of me…
[2] As far as I remember, High Memory is used for application space and Low Memory for the kernel. Advantage is that (user-s…
[3] It may be a huge doc to start, but I think its worth the time youll need to read it : Have look on the Linux-Insides doc…
[4] HIGHMEM is a range of kernels memory space, but it is NOT memory you access but its a place where you put what you want …
[5] The first reference to turn to is Linux Device Drivers (available both online and in book form), particularly chapter 15…


### Build your first RAG Module.

Compose retriever + generator in a `dspy.Module` so optimizers can tune the full pipeline later.

`forward` retrieves passages, formats them as numbered context, then calls `ChainOfThought`.

In [17]:
class RAG(dspy.Module):
    def __init__(self):
        self.respond = dspy.ChainOfThought('context, question -> response')

    def forward(self, question):
        context = search(question).passages
        return self.respond(context=context, question=question)


Let's use the RAG module.


In [18]:
rag = RAG()
rag(question="what are high memory and low memory on linux?")

Prediction(
    reasoning="High memory (HIGHMEM) and low memory (LOWMEM) in Linux relate to how the kernel manages addressable memory in 32-bit architectures. Low memory is directly mapped into the kernel's virtual address space and is always accessible, while high memory is not permanently mapped and requires temporary mapping (using functions like kmap) for access. Low memory typically includes the portion of physical memory that the kernel can address directly at all times, making it simple to access. High memory, on the other hand, consists of memory that cannot be statically mapped into the kernel's address space and requires explicit mapping when the kernel needs to access it. This division is important for managing memory efficiently, especially in systems with more physical memory than can be permanently mapped.",
    response="In Linux, low memory (LOWMEM) refers to the region of physical memory that is permanently mapped into the kernel's address space, making it directly acc

In [19]:
dspy.inspect_history()





[2026-06-20T12:05:49.478003]

System message:

Your input fields are:
1. `context` (str): 
2. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `response` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## response ## ]]
{response}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `context`, `question`, produce the fields `response`.


User message:

[[ ## context ## ]]
[1] «This is relevant to the Linux kernel; Im not sure how any Unix kernel handles this. The High Memory is the segment of memory that user-space programs can address. It cannot touch Low Memory. Low Memory is the segment of memory that the Linux kernel can address directly. If the kernel must access High Memory, it has to map it into its own address space first. There was a patch introduced re

### Phase 2 benchmark — inference

In [ ]:
# row_p3 = benchmark_program(
#     rag,
#     devset=devset,
#     metric=metric,
#     phase="2 RAG + Chroma",
#     mode="inference",
#     lm=lm,
#     num_threads=24,
# )
# benchmark_rows.append(row_p3)
# results_to_frame(benchmark_rows)

## Using a DSPy Optimizer to improve your RAG prompt.

**Phase 3.** We optimize the RAG generation prompt with [MIPROv2](https://arxiv.org/abs/2406.11695) (`auto="light"` — cheaper than `"medium"`).

MIPROv2 compiles better instructions and few-shot demos into `optimized_rag`. It is **not** a magic button — validate on held-out data.

In [ ]:
# history_before_train = len(lm.history)
# t_train_start = time.perf_counter()

# tp = dspy.MIPROv2(
#     metric=metric,
#     auto="light",
#     num_threads=24,
# )

# optimized_rag = tp.compile(
#     RAG(retriever=search),
#     trainset=trainset,
#     max_bootstrapped_demos=2,
#     max_labeled_demos=2,
# )

# train_wall = time.perf_counter() - t_train_start

In [ ]:
# baseline = rag(question="cmd+tab does not work on hidden or minimized windows")
# print(baseline.response)

You are correct that cmd+tab does not work on hidden or minimized windows. To switch back to a minimized app, you must first switch to another application and let it take focus before returning to the minimized one.


In [ ]:
# pred = optimized_rag(question="cmd+tab does not work on hidden or minimized windows")
# print(pred.response)

The Command + Tab shortcut on macOS is designed to switch between currently open applications, but it does not directly restore minimized or hidden windows. When you use Command + Tab, it cycles through the applications that are actively running, and minimized windows do not count as active. To manage minimized windows, you can use other shortcuts or methods. For example, you can use Command + Option + H + M to hide all other applications and minimize the most recently used one. Alternatively, you can navigate to the application you want to restore using Command + Tab and then manually click on the minimized window in the Dock to bring it back to focus.


You can use `dspy.inspect_history(n=2)` to view the RAG prompt [before optimization](https://gist.github.com/okhat/5d04648f2226e72e66e26a8cb1456ee4) and [after optimization](https://gist.github.com/okhat/79405b8889b4b07da577ee19f1a3479a).

Concretely, in one of the runs of this notebook, the optimized prompt does the following (note that it may be different on a later rerun).

1. Constructs the following instruction,
```text
Using the provided `context` and `question`, analyze the information step by step to generate a comprehensive and informative `response`. Ensure that the response clearly explains the concepts involved, highlights key distinctions, and addresses any complexities noted in the context.
```

2. And includes two fully worked out RAG examples with synthetic reasoning and answers, e.g. `how to transfer whatsapp voice message to computer?`.

Let's now evaluate on the overall devset.

### Phase 3 benchmark — train-time

In [ ]:
# row_p2_train = benchmark_program(
#     optimized_rag,
#     devset=devset,
#     metric=metric,
#     phase="3 optimized RAG",
#     mode="train",
#     lm=lm,
#     history_start=history_before_train,
#     wall_seconds=train_wall,
# )
# benchmark_rows.append(row_p2_train)
# results_to_frame(benchmark_rows)

### Phase 3 benchmark — inference

In [ ]:
# row_p2_inf = benchmark_program(
#     optimized_rag,
#     devset=devset,
#     metric=metric,
#     phase="3 optimized RAG",
#     mode="inference",
#     lm=lm,
#     num_threads=24,
# )
# benchmark_rows.append(row_p2_inf)
# results_to_frame(benchmark_rows)

In [ ]:
# print("RAG baseline:\n", rag(question="cmd+tab does not work on hidden or minimized windows").response)
# print()
# print("Optimized RAG:\n", optimized_rag(question="cmd+tab does not work on hidden or minimized windows").response)

## Keeping an eye on cost.

Cumulative view across phases. RAG usually **raises accuracy** but also **latency** (retrieval + longer prompts) and **cost per query** (more input tokens).

Train-time cost is one-off during MIPROv2; inference cost repeats on every user query.

In [ ]:
# summary = results_to_frame(benchmark_rows)
# display(summary)

## Saving and loading.

Save the optimized RAG program so you can reload without re-running MIPROv2.

In [ ]:
# save_path = DATA_DIR / "optimized_rag.json"
# optimized_rag.save(str(save_path))

# loaded_rag = RAG(retriever=search)
# loaded_rag.load(str(save_path))

# loaded_rag(question="cmd+tab does not work on hidden or minimized windows").response

<details>
<summary>Saving programs in MLflow Experiment</summary>

See [MLflow DSPy Documentation](https://mlflow.org/docs/latest/llms/dspy/index.html).

</details>

## What's next?

Improving from a ~40% baseline to ~55%+ with RAG — and further with prompt optimization — is straightforward in DSPy, but we've barely scratched the surface.

Paths to continue:

1. **Better architectures** — e.g. generate search queries before retrieving ([STORM](https://arxiv.org/abs/2402.14207))
2. **Other optimizers** — prompt or weight optimizers ([docs](https://dspy.ai/learn/optimization/optimizers/))
3. **Scale inference compute** — ensembling optimized programs
4. **Cut cost** — distill to a smaller LM

### How do you decide which ones to proceed with first?

The first step is to look at your system outputs, which will allow you to identify the sources of lower performance if any. While doing all of this, make sure you continue to refine your metric, e.g. by optimizing against your judgments, and to collect more (or more realistic) data, e.g. from related domains or from putting a demo of your system in front of users.